# Lecturas y pérdidas

[`07a`](07a_clasificador_nativo.ipynb) entrenó un clasificador con dos etiquetas, una lectura `sign` y la pérdida hinge. La lectura y la pérdida deciden qué tipo de problema resuelve un modelo. Este notebook las cambia para clasificar tres clases, para predecir números reales y para encontrar el estado de menor energía de un hamiltoniano sin ninguna etiqueta, y después compara entrenar con probabilidades exactas y entrenar con shots.

## Varias clases: `argmax`

Con más de dos clases, el readout lleva un observable por clase, y `decision="argmax"` predice el índice del que tiene mayor valor esperado. Aquí hay tres clases, así que tres qubits con `⟨Z₀⟩`, `⟨Z₁⟩` y `⟨Z₂⟩`, y las etiquetas son los índices de clase 0, 1 y 2. El ansatz es `hardware_efficient(1)`, otra disposición estándar con rotaciones `ry` y `rz` en cada qubit y una cadena de puertas `cx`.

La pérdida que le corresponde es `categorical_cross_entropy`: convierte los tres valores esperados en probabilidades y puntúa cuánta probabilidad recibe la clase correcta. El optimizador es el `DE` de [`06a`](06a_entrenamiento_variacional.ipynb), que funciona con modelos nativos igual que con circuitos. La exactitud se mide sobre las mismas nueve muestras de entrenamiento, ya que aquí lo que importa es la lectura:

In [ ]:
import math

import numpy as np
import polypus
from polypus import qml

x_tres = [
    [0.35, 0.40, 0.45],
    [0.45, 0.35, 0.40],
    [0.40, 0.45, 0.35],
    [1.50, 1.55, 1.60],
    [1.60, 1.50, 1.55],
    [1.55, 1.60, 1.50],
    [2.70, 2.75, 2.65],
    [2.65, 2.70, 2.75],
    [2.75, 2.65, 2.70],
]
y_tres = [0, 0, 0, 1, 1, 1, 2, 2, 2]

tres_clases = qml.Model(
    qml.Encoder(3).angle_encoder("ry"),
    qml.Ansatz(3).hardware_efficient(1),
    qml.Readout([qml.Z(0), qml.Z(1), qml.Z(2)], decision="argmax"),
)
result_tres = tres_clases.fit(
    x_tres,
    y_tres,
    # tolerance mínima y patience=60: DE ejecuta las 60 generaciones
    method=polypus.DE(generations=60, population_size=24, tolerance=1e-9, patience=60),
    loss="categorical_cross_entropy",
    exact=True,
    seed=17,
    id="tres_clases",
)
predicciones = tres_clases.bind(result_tres.best_params).predict(x_tres, exact=True)
print("predicciones:", predicciones)
print("exactitud:", np.mean(np.array(predicciones) == y_tres), "| azar: 1/3")

## Regresión: `raw`

Un modelo también puede predecir un número real en vez de una clase. Con `decision="raw"` la predicción es el propio valor esperado `⟨Z₀⟩`, un número entre −1 y 1, y `loss="squared_error"` puntúa la distancia al cuadrado hasta el objetivo. Aquí el objetivo es la curva 0.8·cos x, con una característica sobre un solo qubit. El fitness es el error cuadrático medio con el signo cambiado:

In [ ]:
import matplotlib.pyplot as plt

x_curva = [[0.2 + i * 0.25] for i in range(12)]
y_curva = [0.8 * math.cos(fila[0]) for fila in x_curva]

regresion = qml.Model(
    qml.Encoder(1).angle_encoder("ry"),
    qml.Ansatz(1).hardware_efficient(1),
    qml.Readout([qml.Z(0)], decision="raw"),
)
result_curva = regresion.fit(
    x_curva,
    y_curva,
    method=polypus.Adam(max_iters=200, learning_rate=0.05),
    loss="squared_error",
    exact=True,
    seed=3,
    id="regresion",
)
ajuste = regresion.bind(result_curva.best_params).predict(x_curva, exact=True)
print("error cuadrático medio:", -result_curva.best_fitness)
print("mayor error:", max(abs(p - t) for p, t in zip(ajuste, y_curva)))

puntos = [fila[0] for fila in x_curva]
plt.figure(figsize=(5, 3))
plt.plot(puntos, y_curva, "o", label="objetivo 0.8·cos x")
plt.plot(puntos, ajuste, "x", label="predicción")
plt.xlabel("x")
plt.legend()
plt.show()

## Sin etiquetas: el estado fundamental de un hamiltoniano

Sin etiquetas, `fit` maximiza el valor esperado del primer observable del readout: `loss="expectation"`, el valor por defecto cuando falta `y`. El propio readout pasa a ser el objetivo, lo que convierte el modelo en un resolvedor variacional de autovalores. Para encontrar la menor energía de un hamiltoniano `H`, el readout mide `−H`, escrito como un `qml.Observable` con los coeficientes negados, de modo que maximizar `⟨−H⟩` minimiza la energía `⟨H⟩`.

El hamiltoniano aquí es `H = Z₀Z₁ + 0.5·Z₁` sobre dos qubits. Su menor energía es −1.5, y solo se alcanza en el estado con el qubit 1 en `1` y el qubit 0 en `0`, el bitstring `10` con el qubit 0 a la derecha como en toda la serie. La única muestra es el punto `[0, 0]`: con él, el encoder `ry` no rota nada, así que el ansatz prepara el estado por sí solo:

In [ ]:
menos_h = qml.Observable([(-1.0, qml.Z(0) @ qml.Z(1)), (-0.5, qml.Z(1))])
resolvedor = qml.Model(
    qml.Encoder(2).angle_encoder("ry"),
    qml.Ansatz(2).hardware_efficient(2),
    qml.Readout([menos_h]),
)
result_h = resolvedor.fit(
    [[0.0, 0.0]],
    method=polypus.Adam(max_iters=150, learning_rate=0.1, tolerance=1e-9, patience=150),
    exact=True,
    seed=4,
    id="estado_fundamental",
)
print("energía encontrada:", -result_h.best_fitness, "| menor energía: -1.5")

`run` con `exact=True` devuelve un `ExactResult`, con la probabilidad exacta de cada bitstring para cada fila. El bitstring más probable es el estado que ha preparado el modelo:

In [ ]:
estado = resolvedor.bind(result_h.best_params).run([[0.0, 0.0]], exact=True)
probabilidades = estado.probabilities[0]
mas_probable = max(probabilidades, key=probabilidades.get)
print(
    "estado más probable:", mas_probable, "| probabilidad", probabilidades[mas_probable]
)

## Exacto o muestreado

Todos los entrenamientos hasta aquí han usado `exact=True`: las probabilidades salen directamente del vector de estado, sin ruido de muestreo. Esa opción necesita el simulador nativo local, y su memoria crece como 2ⁿ con el número de qubits `n`, así que es una herramienta de simulación. Un ordenador cuántico real solo da shots, y un modelo entrenado en él estima cada fitness y cada gradiente a partir de ellos. Sin `exact=True`, `fit` entrena así, con `shots` medidas por circuito y la `seed` fijando el muestreo, de modo que la ejecución sigue repitiéndose exactamente igual.

A continuación se entrena el mismo modelo de las dos maneras, sobre dos grupos como los de [`07a`](07a_clasificador_nativo.ipynb) escritos como ocho puntos. Con shots, `best_fitness` es a su vez una estimación, así que los parámetros de las dos ejecuciones se vuelven a puntuar después con `exact=True`:

In [ ]:
x_dos = [
    [0.50, 0.55],
    [0.60, 0.45],
    [0.45, 0.60],
    [0.55, 0.50],
    [2.55, 2.60],
    [2.65, 2.50],
    [2.50, 2.65],
    [2.60, 2.55],
]
y_dos = [-1, -1, -1, -1, 1, 1, 1, 1]

clasificador = qml.Model(
    qml.Encoder(2).angle_encoder("ry"),
    qml.Ansatz(2).hardware_efficient(1),
    qml.Readout([qml.Z(0)], decision="sign"),
)
adam = polypus.Adam(max_iters=40, learning_rate=0.2)
result_exacto = clasificador.fit(
    x_dos, y_dos, method=adam, loss="hinge", exact=True, seed=5, id="exacto"
)
result_shots = clasificador.fit(
    x_dos, y_dos, method=adam, loss="hinge", shots=1024, seed=5, id="shots"
)

for nombre, result in (("exacto", result_exacto), ("1024 shots", result_shots)):
    entrenado = clasificador.bind(result.best_params)
    exactitud = np.mean(np.array(entrenado.predict(x_dos, exact=True)) == y_dos)
    print(
        f"{nombre:>10}: best_fitness {result.best_fitness:.4f}, exactitud {exactitud}"
    )

Un modelo entrenado también se ejecuta de las dos maneras. Sin `exact=True`, `run` devuelve un `RunResult` con los counts medidos para cada fila, el mismo tipo de resultado que `run_quantum_circuit` en [`03a`](03a_ejecucion_local.ipynb). Con él, `run` devuelve un `ExactResult` con las probabilidades. Divididos entre el número de shots, los counts se acercan a esas probabilidades:

In [ ]:
entrenado = clasificador.bind(result_exacto.best_params)
muestra = x_dos[:1]
counts = entrenado.run(muestra, shots=1000, seed=3).counts[0]
probabilidades = entrenado.run(muestra, exact=True).probabilities[0]
for bits in sorted(probabilidades):
    print(
        bits,
        "| counts / 1000:",
        counts.get(bits, 0) / 1000,
        "| exacto:",
        round(probabilidades[bits], 3),
    )

## Resumen

Este notebook ha cambiado la lectura y la pérdida de un modelo nativo: `argmax` con `categorical_cross_entropy` para varias clases, `raw` con `squared_error` para regresión, y un readout sin etiquetas que encuentra el estado fundamental de un hamiltoniano. También ha entrenado el mismo modelo con probabilidades exactas y con shots, y lo ha ejecutado de las dos maneras con `run`.

## Fin de la serie

Esta serie ha recorrido el camino completo: desde el primer circuito hasta entrenar modelos con datos. Quien quiera seguir puede consultar [`examples/`](https://github.com/Bahia-Software/polypus/tree/main/examples), con casos más grandes, entre ellos una versión completa de MaxCut con QAOA. Dos de ellos continúan esta sección: `qml_iris_demo.py` entrena un clasificador nativo con las flores Iris, y `qml_native_showcase.py` recorre todos los encoders, capas, optimizadores, lecturas y pérdidas nativos. La [guía de usuario](https://github.com/Bahia-Software/polypus/tree/main/docs/guide), en inglés, trata a fondo el entrenamiento, QML, la memoria y el rendimiento.

Volver al índice: [`00_bienvenida.ipynb`](00_bienvenida.ipynb).